# Cenário 6: Relação Exponencial com Erro Multiplicativo Lognormal - Simulação MRLS

## Objetivo
Estudar o comportamento do MQO quando a relação verdadeira entre resposta e explicativa é multiplicativa e, após transformação logarítmica, passa a ser linear.

## Especificação do Cenário
- Na escala original: $Y_i = xp(eta_0 + eta_1 X_i)ta_i$
- Na escala logarítmica: $og(Y_i) = eta_0 + eta_1 X_i + psilon_i$
- O ajuste será comparado em duas versões: modelo original e modelo transformado em log

## Estrutura do Estudo
Seguiremos a mesma lógica dos notebooks anteriores:
1. Definição dos parâmetros
2. Geração de $X$ e de $Y$
3. Ajuste do modelo em duas escalas
4. Repetição da simulação
5. Resumo das métricas
6. Visualizações e diagnóstico

---

## Algoritmo de Simulação

In [ ]:
# 1. Importar bibliotecas
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from tqdm import tqdm

warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

### Passo 1: Definir parâmetros do modelo
Usamos valores simples e interpretáveis, como recomendado no manual, e uma variância lognormal moderada para que a comparação entre escalas fique visível.

In [ ]:
# 2. Configuração de parâmetros
BETA0_TRUE = 1.0
BETA1_TRUE = 2.0
LOGNORMAL_SIGMA = 0.4
SAMPLE_SIZES = [30, 100]
NUM_REPLICATIONS = 1000
ALPHA = 0.05
SEED = 20260506
X_LOW = -2.0
X_HIGH = 2.0
OUTPUT_DIR = 'cenario6_resultados'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Parâmetros do Cenário 6: relação exponencial com erro multiplicativo lognormal')
print(f'  beta0 = {BETA0_TRUE}')
print(f'  beta1 = {BETA1_TRUE}')
print(f'  sigma lognormal = {LOGNORMAL_SIGMA}')
print(f'  n = {SAMPLE_SIZES}')
print(f'  B = {NUM_REPLICATIONS}')
print(f'  X ~ Uniforme({X_LOW}, {X_HIGH})')

#### Funções utilitárias
Aqui criamos a geração do cenário, o ajuste OLS nas duas escalas e as funções de resumo das repetições.

In [ ]:
# 3. Funções utilitárias
def generate_X(n, rng, x_low, x_high):
    """Passo 2: Gera X uniformemente distribuído ou carrega CSV existente.
    Se existir 'X_n{n}.csv' no diretório atual, carrega-o; caso contrário, gera e salva."""
    filename = f"X_n{n}.csv"
    # Se o arquivo existir, tentar carregar e retornar os valores
    if os.path.exists(filename):
        try:
            df = pd.read_csv(filename, header=None)
            return df.iloc[:, 0].to_numpy()
        except Exception as e:
            print(f"Aviso: falha ao ler {filename}: {e}. Gerando novo X.")

def generate_Y_scenario6(X, beta0, beta1, lognormal_sigma, rng):
    eps = rng.normal(loc=0.0, scale=lognormal_sigma, size=len(X))
    eta = np.exp(eps)
    Y = np.exp(beta0 + beta1 * X) * eta
    return Y, eps, eta

def fit_ols_model(X, Y, alpha):
    X_with_const = sm.add_constant(X, has_constant='add')
    model = sm.OLS(Y, X_with_const).fit()
    ci = model.conf_int(alpha=alpha)
    return {
        'beta0_hat': float(model.params[0]),
        'beta1_hat': float(model.params[1]),
        'se_beta0': float(model.bse[0]),
        'se_beta1': float(model.bse[1]),
        'ci_beta0_low': float(ci[0, 0]),
        'ci_beta0_high': float(ci[0, 1]),
        'ci_beta1_low': float(ci[1, 0]),
        'ci_beta1_high': float(ci[1, 1]),
        'pvalue_beta1': float(model.pvalues[1]),
        'resid': np.array(model.resid),
        'fitted': np.array(model.fittedvalues),
        'r_squared': float(model.rsquared),
    }

def fit_log_model(X, Y, alpha):
    log_y = np.log(Y)
    result = fit_ols_model(X, log_y, alpha)
    result['log_y'] = log_y
    return result

def summarize_estimator(estimates, true_value):
    mean_est = np.mean(estimates)
    bias = mean_est - true_value
    variance = np.var(estimates, ddof=1)
    mse = np.mean((estimates - true_value) ** 2)
    return {'mean': mean_est, 'bias': bias, 'variance': variance, 'mse': mse, 'std_dev': np.sqrt(variance)}

def compute_coverage(ci_low, ci_high, true_value):
    return float(np.mean((ci_low <= true_value) & (true_value <= ci_high)))

def compute_rejection_rate(pvalues, alpha):
    return float(np.mean(pvalues < alpha))

def run_replications(X, rng):
    original_rows = []
    log_rows = []
    diagnostic_original = None
    diagnostic_log = None

    for rep in tqdm(range(NUM_REPLICATIONS), desc='Repetições'):
        Y, eps, eta = generate_Y_scenario6(X, BETA0_TRUE, BETA1_TRUE, LOGNORMAL_SIGMA, rng)
        fit_original = fit_ols_model(X, Y, ALPHA)
        fit_log = fit_log_model(X, Y, ALPHA)

        original_rows.append({
            'beta0_hat': fit_original['beta0_hat'],
            'beta1_hat': fit_original['beta1_hat'],
            'se_beta0': fit_original['se_beta0'],
            'se_beta1': fit_original['se_beta1'],
            'ci_beta0_low': fit_original['ci_beta0_low'],
            'ci_beta0_high': fit_original['ci_beta0_high'],
            'ci_beta1_low': fit_original['ci_beta1_low'],
            'ci_beta1_high': fit_original['ci_beta1_high'],
            'pvalue_beta1': fit_original['pvalue_beta1'],
            'r_squared': fit_original['r_squared'],
        })

        log_rows.append({
            'beta0_hat': fit_log['beta0_hat'],
            'beta1_hat': fit_log['beta1_hat'],
            'se_beta0': fit_log['se_beta0'],
            'se_beta1': fit_log['se_beta1'],
            'ci_beta0_low': fit_log['ci_beta0_low'],
            'ci_beta0_high': fit_log['ci_beta0_high'],
            'ci_beta1_low': fit_log['ci_beta1_low'],
            'ci_beta1_high': fit_log['ci_beta1_high'],
            'pvalue_beta1': fit_log['pvalue_beta1'],
            'r_squared': fit_log['r_squared'],
        })

        if diagnostic_original is None:
            diagnostic_original = {
                'fitted': np.array(fit_original['fitted']),
                'resid': np.array(fit_original['resid']),
                'Y': Y,
                'logY': np.log(Y),
            }
            diagnostic_log = {
                'fitted': np.array(fit_log['fitted']),
                'resid': np.array(fit_log['resid']),
                'logY': np.log(Y),
            }

    return pd.DataFrame(original_rows), pd.DataFrame(log_rows), diagnostic_original, diagnostic_log

### Passos 2-4: Gerar X uma única vez e preparar a simulação
Para manter comparabilidade, a mesma amostra de X é usada em todas as repetições dentro de cada tamanho amostral.

In [ ]:
# 4. Simulação principal
rng = np.random.default_rng(SEED)
results = {}

for n in SAMPLE_SIZES:
    X = generate_X(n, rng, X_LOW, X_HIGH)

    print('\n' + '='*60)
    print(f'Simulação para n = {n}')
    print('='*60)

    original_frame, log_frame, diag_original, diag_log = run_replications(X, rng)

    results[n] = {
        'X': X,
        'original_frame': original_frame,
        'log_frame': log_frame,
        'original_beta0': original_frame['beta0_hat'].to_numpy(),
        'original_beta1': original_frame['beta1_hat'].to_numpy(),
        'original_se_beta0': original_frame['se_beta0'].to_numpy(),
        'original_se_beta1': original_frame['se_beta1'].to_numpy(),
        'original_ci_beta0_low': original_frame['ci_beta0_low'].to_numpy(),
        'original_ci_beta0_high': original_frame['ci_beta0_high'].to_numpy(),
        'original_ci_beta1_low': original_frame['ci_beta1_low'].to_numpy(),
        'original_ci_beta1_high': original_frame['ci_beta1_high'].to_numpy(),
        'original_pvalue_beta1': original_frame['pvalue_beta1'].to_numpy(),
        'original_r_squared': original_frame['r_squared'].to_numpy(),
        'log_beta0': log_frame['beta0_hat'].to_numpy(),
        'log_beta1': log_frame['beta1_hat'].to_numpy(),
        'log_se_beta0': log_frame['se_beta0'].to_numpy(),
        'log_se_beta1': log_frame['se_beta1'].to_numpy(),
        'log_ci_beta0_low': log_frame['ci_beta0_low'].to_numpy(),
        'log_ci_beta0_high': log_frame['ci_beta0_high'].to_numpy(),
        'log_ci_beta1_low': log_frame['ci_beta1_low'].to_numpy(),
        'log_ci_beta1_high': log_frame['ci_beta1_high'].to_numpy(),
        'log_pvalue_beta1': log_frame['pvalue_beta1'].to_numpy(),
        'log_r_squared': log_frame['r_squared'].to_numpy(),
        'diagnostic_original': diag_original,
        'diagnostic_log': diag_log,
    }

    print(f'✓ Simulação concluída para n = {n}')

print('\n' + '='*60)
print('Todas as simulações foram concluídas com sucesso!')
print('='*60)

## Etapas do Trabalho

### Etapa 1: Estimação
Compararemos os estimadores nas duas escalas. A expectativa é que o ajuste na escala logarítmica seja mais estável e interpretable, porque é a forma correta do modelo gerador.

In [ ]:
# Cálculo de estatísticas de estimação
estimation_summary = []

for n in SAMPLE_SIZES:
    summary_original_beta0 = summarize_estimator(results[n]['original_beta0'], np.exp(BETA0_TRUE))
    summary_original_beta1 = summarize_estimator(results[n]['original_beta1'], BETA1_TRUE)
    summary_log_beta0 = summarize_estimator(results[n]['log_beta0'], BETA0_TRUE)
    summary_log_beta1 = summarize_estimator(results[n]['log_beta1'], BETA1_TRUE)

    estimation_summary.append({'n': n, 'modelo': 'escala original', 'parâmetro': 'beta0', 'valor_verdadeiro': np.exp(BETA0_TRUE), 'média_estimador': summary_original_beta0['mean'], 'viés': summary_original_beta0['bias'], 'variância': summary_original_beta0['variance'], 'desvio_padrão': summary_original_beta0['std_dev'], 'EQM': summary_original_beta0['mse']})
    estimation_summary.append({'n': n, 'modelo': 'escala original', 'parâmetro': 'beta1', 'valor_verdadeiro': BETA1_TRUE, 'média_estimador': summary_original_beta1['mean'], 'viés': summary_original_beta1['bias'], 'variância': summary_original_beta1['variance'], 'desvio_padrão': summary_original_beta1['std_dev'], 'EQM': summary_original_beta1['mse']})
    estimation_summary.append({'n': n, 'modelo': 'log(Y)', 'parâmetro': 'beta0', 'valor_verdadeiro': BETA0_TRUE, 'média_estimador': summary_log_beta0['mean'], 'viés': summary_log_beta0['bias'], 'variância': summary_log_beta0['variance'], 'desvio_padrão': summary_log_beta0['std_dev'], 'EQM': summary_log_beta0['mse']})
    estimation_summary.append({'n': n, 'modelo': 'log(Y)', 'parâmetro': 'beta1', 'valor_verdadeiro': BETA1_TRUE, 'média_estimador': summary_log_beta1['mean'], 'viés': summary_log_beta1['bias'], 'variância': summary_log_beta1['variance'], 'desvio_padrão': summary_log_beta1['std_dev'], 'EQM': summary_log_beta1['mse']})

estimation_df = pd.DataFrame(estimation_summary)
print('\n' + '='*120)
print('TABELA 1: ESTATISTICAS DE ESTIMACAO')
print('='*120)
print(estimation_df.to_string(index=False))
print('='*120)

all_results = {'estimation': estimation_df}

### Etapa 2: Inferência
Vamos comparar a cobertura empírica dos intervalos de confiança nas duas escalas. O ajuste em log deve apresentar desempenho mais alinhado com a teoria do MRLS.

In [ ]:
# Cálculo de cobertura dos intervalos de confiança
inference_summary = []

for n in SAMPLE_SIZES:
    cov_original_beta0 = compute_coverage(results[n]['original_ci_beta0_low'], results[n]['original_ci_beta0_high'], np.exp(BETA0_TRUE))
    cov_original_beta1 = compute_coverage(results[n]['original_ci_beta1_low'], results[n]['original_ci_beta1_high'], BETA1_TRUE)
    cov_log_beta0 = compute_coverage(results[n]['log_ci_beta0_low'], results[n]['log_ci_beta0_high'], BETA0_TRUE)
    cov_log_beta1 = compute_coverage(results[n]['log_ci_beta1_low'], results[n]['log_ci_beta1_high'], BETA1_TRUE)

    inference_summary.append({'n': n, 'modelo': 'escala original', 'parâmetro': 'beta0', 'cobertura_empírica': cov_original_beta0, 'cobertura_teórica': 1 - ALPHA, 'diferença': cov_original_beta0 - (1 - ALPHA)})
    inference_summary.append({'n': n, 'modelo': 'escala original', 'parâmetro': 'beta1', 'cobertura_empírica': cov_original_beta1, 'cobertura_teórica': 1 - ALPHA, 'diferença': cov_original_beta1 - (1 - ALPHA)})
    inference_summary.append({'n': n, 'modelo': 'log(Y)', 'parâmetro': 'beta0', 'cobertura_empírica': cov_log_beta0, 'cobertura_teórica': 1 - ALPHA, 'diferença': cov_log_beta0 - (1 - ALPHA)})
    inference_summary.append({'n': n, 'modelo': 'log(Y)', 'parâmetro': 'beta1', 'cobertura_empírica': cov_log_beta1, 'cobertura_teórica': 1 - ALPHA, 'diferença': cov_log_beta1 - (1 - ALPHA)})

inference_df = pd.DataFrame(inference_summary)
print('\n' + '='*120)
print('TABELA 2: COBERTURA EMPIRICA DOS INTERVALOS DE CONFIANCA')
print(f'(Nível nominal = {100 * (1 - ALPHA):.1f}%)')
print('='*120)
print(inference_df.to_string(index=False))
print('='*120)

all_results['inference'] = inference_df

### Etapa 3: Testes de Hipóteses
A taxa de rejeição também é comparada nas duas escalas. Na escala logarítmica, a inferência deve ser mais adequada ao mecanismo gerador do dado.

In [ ]:
tests_summary = []

for n in SAMPLE_SIZES:
    power_original = compute_rejection_rate(results[n]['original_pvalue_beta1'], ALPHA)
    power_log = compute_rejection_rate(results[n]['log_pvalue_beta1'], ALPHA)

    tests_summary.append({'n': n, 'modelo': 'escala original', 'hipótese': 'H1: beta1 diferente de 0', 'taxa_rejeição': power_original, 'interpretação': 'Poder do teste na escala original'})
    tests_summary.append({'n': n, 'modelo': 'log(Y)', 'hipótese': 'H1: beta1 diferente de 0', 'taxa_rejeição': power_log, 'interpretação': 'Poder do teste na escala log'})

tests_df = pd.DataFrame(tests_summary)
print('\n' + '='*100)
print('TABELA 3: TESTES DE HIPOTESES')
print(f'(Nível de significância = {100 * ALPHA:.1f}%)')
print('='*100)
print(tests_df.to_string(index=False))
print('='*100)

all_results['tests'] = tests_df

### Etapa 4: Diagnóstico e Visualizações
O diagnóstico precisa mostrar duas mensagens: na escala original o padrão dos resíduos tende a ser menos comportado, enquanto na escala logarítmica a estrutura deve parecer mais compatível com o MRLS.

In [ ]:
# 4.1 Visualizações dos estimadores: beta1
fig, axes = plt.subplots(len(SAMPLE_SIZES), 3, figsize=(15, 5 * len(SAMPLE_SIZES)))
if len(SAMPLE_SIZES) == 1:
    axes = axes.reshape(1, -1)
for idx, n in enumerate(SAMPLE_SIZES):
    ax = axes[idx, 0]
    samples = results[n]['log_beta1']
    ax.hist(samples, bins=40, density=True, alpha=0.7, color='steelblue', edgecolor='black')
    mu = np.mean(samples)
    sd = np.std(samples)
    grid = np.linspace(mu - 4 * sd, mu + 4 * sd, 100)
    ax.plot(grid, stats.norm.pdf(grid, mu, sd), 'r-', linewidth=2, label='Normal')
    ax.axvline(BETA1_TRUE, color='green', linestyle='--', linewidth=2, label=f'beta1 verdadeiro = {BETA1_TRUE}')
    ax.set_title(f'Histograma de beta1 em log(Y) (n={n})')
    ax.legend()
    ax = axes[idx, 1]
    ax.boxplot([samples], labels=[f'n={n}'], vert=True)
    ax.axhline(BETA1_TRUE, color='green', linestyle='--', linewidth=2)
    ax.set_title(f'Boxplot de beta1 em log(Y) (n={n})')
    ax = axes[idx, 2]
    stats.probplot(samples, dist='norm', plot=ax)
    ax.set_title(f'QQ-plot de beta1 em log(Y) (n={n})')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cenario6_diagnostico_beta1_log.png'), dpi=100, bbox_inches='tight')
plt.show()

# 4.2 Diagnóstico de resíduos: escala original
n_diag = max(SAMPLE_SIZES)
resid_original = results[n_diag]['diagnostic_original']['resid']
fitted_original = results[n_diag]['diagnostic_original']['fitted']
resid_log = results[n_diag]['diagnostic_log']['resid']
fitted_log = results[n_diag]['diagnostic_log']['fitted']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
ax = axes[0, 0]
ax.scatter(fitted_original, resid_original, alpha=0.6, s=50, color='steelblue', edgecolor='darkblue', linewidth=0.5)
ax.axhline(0, color='red', linestyle='--', linewidth=2)
ax.set_title(f'Resíduos vs ajustados - escala original (n={n_diag})')
ax.set_xlabel('Valores ajustados')
ax.set_ylabel('Resíduos')

ax = axes[0, 1]
stats.probplot(resid_original, dist='norm', plot=ax)
ax.set_title('QQ-plot dos resíduos - escala original')

ax = axes[1, 0]
ax.scatter(fitted_log, resid_log, alpha=0.6, s=50, color='steelblue', edgecolor='darkblue', linewidth=0.5)
ax.axhline(0, color='red', linestyle='--', linewidth=2)
ax.set_title(f'Resíduos vs ajustados - log(Y) (n={n_diag})')
ax.set_xlabel('Valores ajustados')
ax.set_ylabel('Resíduos')

ax = axes[1, 1]
stats.probplot(resid_log, dist='norm', plot=ax)
ax.set_title('QQ-plot dos resíduos - log(Y)')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cenario6_diagnostico_residuos.png'), dpi=100, bbox_inches='tight')
plt.show()

print('\nInterpretação:')
print('  - Na escala original, o ajuste linear em Y tende a não capturar bem a estrutura multiplicativa.')
print('  - Na escala logarítmica, a relação volta à forma linear e os resíduos ficam mais adequados.')

all_results['diagnostics'] = pd.DataFrame([{'n': n_diag, 'descricao': 'Diagnostico qualitativo realizado com escala original e log'}])

## Resumo e Interpretação Final

1. **Escala original**: o ajuste linear direto em Y tende a ser menos apropriado porque a relação verdadeira é multiplicativa.
2. **Escala logarítmica**: ao transformar Y em log(Y), o modelo passa a coincidir com o mecanismo gerador e a inferência melhora.
3. **Diagnóstico**: os resíduos na escala log costumam apresentar comportamento mais compatível com as hipóteses do MRLS.
4. **Conclusão**: este cenário mostra por que transformações podem ser essenciais para obter um modelo linear mais adequado e uma inferência mais confiável.

In [ ]:
# Salvar resultados em CSV
print('\n' + '='*100)
print('SALVANDO RESULTADOS')
print('='*100)
for name, df in all_results.items():
    filepath = os.path.join(OUTPUT_DIR, f'cenario6_{name}.csv')
    df.to_csv(filepath, index=False)
    print(f'✓ Salvo: {filepath}')
print(f'\n✓ Arquivos salvos no diretório: {OUTPUT_DIR}/')
print('='*100)